# MeshVTON — Inference

Eğitilmiş Aşama-1 checkpoint'iyle: kullanıcının yüklediği KENDİ fotoğrafı + `data/garments_3d/upper_body`
kütüphanesinden seçilen bir 3D giysi → try-on sonucu.

**Not:** Model yalnızca 3D mesh + gri (texture'sız) koşullama ile eğitildi (bkz. eğitim notebook'u,
KALICI no-texture kuralı). Giysi seçimi bu yüzden bir FOTOĞRAF değil, mevcut mesh kütüphanesinden bir
klasör adı (`GARMENT_ID`) — kullanıcı yalnızca KENDİ fotoğrafını sağlıyor.

In [ ]:
#@title 1) Setup (same as the train notebook — pyrender/HMR2/IDM-VTON preprocess)
import os
if not os.path.exists('/content/MeshVTON'):
    !git clone https://github.com/SerhanTelatar/MeshVTON /content/MeshVTON
%cd /content/MeshVTON
!git pull

!pip -q install "diffusers>=0.34" "peft>=0.14" lpips einops sentencepiece trimesh smplx pyrender onnxruntime
!pip -q uninstall -y pyopengl PyOpenGL-accelerate > /dev/null 2>&1
!pip -q install "git+https://github.com/mmatl/pyopengl.git"
import importlib.util
if importlib.util.find_spec('hmr2') is None:
    !pip -q install "git+https://github.com/shubham-goel/4D-Humans.git"
!apt-get -qq install -y libglu1-mesa libosmesa6 > /dev/null 2>&1
import subprocess
_probe = subprocess.run(["python", "-c",
    'import os;os.environ["PYOPENGL_PLATFORM"]="egl";'
    'import pyrender;r=pyrender.OffscreenRenderer(16,16);r.delete();print("egl-ok")'],
    capture_output=True, text=True)
os.environ['PYOPENGL_PLATFORM'] = 'egl' if 'egl-ok' in _probe.stdout else 'osmesa'
print('GL platform:', os.environ['PYOPENGL_PLATFORM'])

if not os.path.exists('/content/IDM-VTON'):
    !git clone -q https://github.com/yisol/IDM-VTON /content/IDM-VTON
import shutil
from huggingface_hub import hf_hub_download
for repo_path in ('humanparsing/parsing_atr.onnx',
                  'humanparsing/parsing_lip.onnx',
                  'openpose/ckpts/body_pose_model.pth'):
    local = f'/content/IDM-VTON/ckpt/{repo_path}'
    if not (os.path.exists(local) and os.path.getsize(local) > 1_000_000):
        os.makedirs(os.path.dirname(local), exist_ok=True)
        shutil.copy(hf_hub_download('yisol/IDM-VTON', repo_path), local)
    assert os.path.getsize(local) > 1_000_000, f'bozuk indirme: {local}'

from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('Kurulum OK')

In [ ]:
#@title 2) Data + checkpoint — the Drive/MeshVTON layout
import os, sys
sys.path.insert(0, '/content/MeshVTON/v2')
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/MeshVTON'

# Giysi kütüphanesi (yalnız upper_body kullanılıyor — model bunlarla eğitildi)
!mkdir -p /content/MeshVTON/data
!unzip -q -n $D/garments_3d.zip -d /content/MeshVTON/data   # -> data/garments_3d/

# SMPL-X gövde modeli
!mkdir -p /content/MeshVTON/checkpoints/pretrained/smplx
!cp $D/smplx/SMPLX_NEUTRAL.* /content/MeshVTON/checkpoints/pretrained/smplx/
os.environ['SMPLX_MODEL_DIR'] = '/content/MeshVTON/checkpoints/pretrained/smplx'

# HMR2 ağırlıkları + SMPL neutral pkl
import glob, shutil
from meshvton2.conditioning.body import _patch_torch_load_weights_only
import torch as _torch; _patch_torch_load_weights_only(_torch)
from hmr2.models import download_models
from hmr2.configs import CACHE_DIR_4DHUMANS
download_models(CACHE_DIR_4DHUMANS)
smpl_dir = f"{CACHE_DIR_4DHUMANS}/data/smpl"; os.makedirs(smpl_dir, exist_ok=True)
cands = glob.glob(f'{D}/smpl/*neutral*lbs*.pkl') + glob.glob(f'{D}/smpl/SMPL_NEUTRAL.pkl')
assert cands, "SMPL neutral pkl yok -> Drive/MeshVTON/smpl/"
shutil.copy(cands[0], f"{smpl_dir}/SMPL_NEUTRAL.pkl")

# Eğitilmiş checkpoint — istenirse değiştirin
CHECKPOINT = f"{D}/v2_outputs/stage1/final.pt"  #@param {type:"string"}
assert os.path.exists(CHECKPOINT), f"checkpoint bulunamadı: {CHECKPOINT}"
print('Veri + checkpoint hazır:', CHECKPOINT)

In [ ]:
#@title 3) Upload the person photos (you may select SEVERAL)
from google.colab import files
from PIL import Image
import pathlib

uploaded = files.upload()  # Ctrl/Cmd ile birden çok dosya seçin
PERSON_IMAGES = [f"/content/{n}" for n in uploaded]
assert PERSON_IMAGES, "hiç dosya yüklenmedi"

# ÖNEMLİ: ön-işleme fotoğrafı KOŞULSUZ 768x1024'e resize eder (person.py:119) —
# en-boy oranı KORUNMAZ, kırpma da yapılmaz. 3:4 dışındaki foto GERİLİR; gerilmiş
# gövdede HMR2 pozu ve parser maskesi bozulur, sonuç kötüleşir. Aşağıdaki uyarıya bakın.
print(f"{len(PERSON_IMAGES)} fotoğraf yüklendi:\n")
for p in PERSON_IMAGES:
    w, h = Image.open(p).size
    ar = w / h
    ok = abs(ar - 0.75) < 0.03
    note = "OK (3:4)" if ok else f"DİKKAT: oran {ar:.2f}, 3:4=0.75 → GERİLECEK, önce kırpın"
    if w < 768 or h < 1024:
        note += "  |  düşük çözünürlük (<768x1024, büyütülecek)"
    print(f"  {pathlib.Path(p).name:35s} {w}x{h}  {note}")

In [ ]:
#@title 4) Choose the garment(s) — several, comma-separated
import pathlib
GARMENTS_ROOT = pathlib.Path('/content/MeshVTON/data/garments_3d')
available = sorted(p.parent for p in (GARMENTS_ROOT / 'upper_body').rglob('*.obj'))
print(f'{len(available)} giysi bulundu, ilk 10:')
for d in available[:10]:
    print(' ', d.relative_to(GARMENTS_ROOT))

GARMENT_IDS = "upper_body/00047_Top, upper_body/00111_Tshirt"  #@param {type:"string"}
GARMENT_LIST = [g.strip() for g in GARMENT_IDS.split(',') if g.strip()]
missing = [g for g in GARMENT_LIST if not (GARMENTS_ROOT / g).exists()]
assert not missing, f'giysi klasörü yok: {missing}'
print(f'\n{len(GARMENT_LIST)} giysi seçildi: {GARMENT_LIST}')
print(f'toplam üretilecek: {len(PERSON_IMAGES)} foto × {len(GARMENT_LIST)} giysi = '
      f'{len(PERSON_IMAGES) * len(GARMENT_LIST)} görsel (~{len(PERSON_IMAGES) * len(GARMENT_LIST) * 19} sn)')

In [ ]:
#@title 5) Generate
import sys, yaml, numpy as np
sys.path.insert(0, '/content/MeshVTON/v2')
from meshvton2.conditioning.body import build_hmr2_backend
from meshvton2.conditioning.builder import PhotoView, assert_real_impl, build_conditioning
from meshvton2.conditioning.garment import load_garment_asset
from meshvton2.conditioning.person import PersonPreprocessor, person_square_bbox
from meshvton2.model.flux_tryon import FluxTryOnSampler

assert_real_impl()
base = yaml.safe_load(open('/content/MeshVTON/v2/configs/base.yaml'))
size = (base['resolution']['height'], base['resolution']['width'])

prep = PersonPreprocessor('/content/IDM-VTON')
hmr2 = build_hmr2_backend()
pp = prep.process(PERSON_IMAGE, size=size)
params = hmr2(pp.image, bbox=person_square_bbox(pp))  # kişi-merkezli kare bbox (hizalama düzeltmesi)

mesh = sorted(GARMENT_DIR.glob('*.obj'))[0]
asset = load_garment_asset(mesh, garment_id=GARMENT_ID.replace('/', '__'), allow_untextured=True)

bundle = build_conditioning(pp.image, params, asset, PhotoView(), size=size, person_prep=pp)

sampler = FluxTryOnSampler(base['model']['flux_fill_repo'], checkpoint=CHECKPOINT,
                            prompt=base['model']['prompt'])
result = sampler.sample(bundle, steps=28, seed=0, control_scale=1.0)  # control_scale=1.0: gerçek/dağıtım modu

from PIL import Image
from IPython.display import display
print('Kişi (girdi)              Sonuç')
display(Image.fromarray(pp.image).resize((384, 512)))
display(Image.fromarray(result).resize((384, 512)))